In [109]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm 
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_val_score


Note: LinearRegression/Lasso/Elastic Net/Ridge did not make sense for the fraud use case (binary classification). So I instead use Logistic Reg with different penalty

In [110]:
random_state=42

In [111]:
classification_reports = []
classification_report_keys = []

## Data

In [112]:
# Import IBM dataset
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [113]:
df.drop(columns=['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'], inplace=True)

## Undersampling

In [114]:
from collections import Counter


X = df.drop(columns='Is Laundering')
y = df['Is Laundering']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 5073168, 1: 5177})
Resampled dataset shape: Counter({0: 5177, 1: 5177})


## Split Data

In [115]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

## Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

### Linear Regression

In [116]:
model = LogisticRegression()

In [117]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [118]:
y_pred = model.predict(X_test)

In [119]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logistic Regression')

              precision    recall  f1-score   support

           0       1.00      0.01      0.02      1036
           1       0.50      1.00      0.67      1035

    accuracy                           0.50      2071
   macro avg       0.75      0.50      0.34      2071
weighted avg       0.75      0.50      0.34      2071



### Ridge

In [120]:
from sklearn.linear_model import RidgeClassifier


ridge_model = RidgeClassifier(random_state=random_state)

In [121]:
ridge_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.80134e-23): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,class_weight,None
,solver,'auto'
,positive,False
,random_state,42


In [122]:
y_pred = ridge_model.predict(X_test)

In [123]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Ridge Classifier')


              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



### Lasso

In [124]:
lasso_model = LogisticRegression(penalty='l1', solver='liblinear')

In [125]:
lasso_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [126]:
y_pred = lasso_model.predict(X_test)

In [127]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline L1 (Lasso) Logistic Regression')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



## Parameter Tuning

### Logistic Regression

In [128]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [129]:
# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5)

In [130]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_recall_optuna_results)

[I 2026-05-27 19:18:30,545] A new study created in memory with name: no-name-012426de-af03-4fb9-9015-e2cd1939941b
[I 2026-05-27 19:18:30,699] Trial 0 finished with value: 0.8038647342995169 and parameters: {'C': 0.5123388053070168, 'Class_weight': None}. Best is trial 0 with value: 0.8038647342995169.
[I 2026-05-27 19:18:30,853] Trial 1 finished with value: 0.8038647342995169 and parameters: {'C': 0.32652364507662274, 'Class_weight': None}. Best is trial 0 with value: 0.8038647342995169.
[I 2026-05-27 19:18:31,001] Trial 2 finished with value: 0.8038647342995169 and parameters: {'C': 0.7982390744089412, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8038647342995169.
[I 2026-05-27 19:18:31,155] Trial 3 finished with value: 0.8038647342995169 and parameters: {'C': 0.6334500702856152, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8038647342995169.
[I 2026-05-27 19:18:31,306] Trial 4 finished with value: 0.8038647342995169 and parameters: {'C': 0.6580667355328703,

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
0,0,0.803865,2026-05-27 19:18:30.546431,2026-05-27 19:18:30.699985,0 days 00:00:00.153554,0.512339,None,COMPLETE
1,1,0.803865,2026-05-27 19:18:30.699985,2026-05-27 19:18:30.853027,0 days 00:00:00.153042,0.326524,None,COMPLETE
2,2,0.803865,2026-05-27 19:18:30.853027,2026-05-27 19:18:31.001247,0 days 00:00:00.148220,0.798239,balanced,COMPLETE
3,3,0.803865,2026-05-27 19:18:31.001247,2026-05-27 19:18:31.155919,0 days 00:00:00.154672,0.633450,balanced,COMPLETE
4,4,0.803865,2026-05-27 19:18:31.156488,2026-05-27 19:18:31.306927,0 days 00:00:00.150439,0.658067,None,COMPLETE


In [131]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_f1_optuna_results)

[I 2026-05-27 19:18:47,508] A new study created in memory with name: no-name-96110a97-bfa9-4bb2-b0da-d13b6f351a38
[I 2026-05-27 19:18:47,673] Trial 0 finished with value: 0.5420800717047658 and parameters: {'C': 0.2444207789555405, 'Class_weight': None}. Best is trial 0 with value: 0.5420800717047658.
[I 2026-05-27 19:18:47,832] Trial 1 finished with value: 0.5421129799729683 and parameters: {'C': 0.6852082049021077, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.5421129799729683.
[I 2026-05-27 19:18:48,032] Trial 2 finished with value: 0.5421129799729683 and parameters: {'C': 0.4576910743595539, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.5421129799729683.
[I 2026-05-27 19:18:48,196] Trial 3 finished with value: 0.5421129799729683 and parameters: {'C': 0.6338126414247635, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.5421129799729683.
[I 2026-05-27 19:18:48,354] Trial 4 finished with value: 0.5420800717047658 and parameters: {'C': 0.594645863629

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
67,67,0.543555,2026-05-27 19:18:58.997328,2026-05-27 19:18:59.179551,0 days 00:00:00.182223,0.624296,None,COMPLETE
76,76,0.542130,2026-05-27 19:19:00.566622,2026-05-27 19:19:00.772235,0 days 00:00:00.205613,0.069094,None,COMPLETE
68,68,0.542121,2026-05-27 19:18:59.180551,2026-05-27 19:18:59.337046,0 days 00:00:00.156495,0.677212,None,COMPLETE
1,1,0.542113,2026-05-27 19:18:47.674277,2026-05-27 19:18:47.832973,0 days 00:00:00.158696,0.685208,balanced,COMPLETE
35,35,0.542113,2026-05-27 19:18:53.511332,2026-05-27 19:18:53.677268,0 days 00:00:00.165936,0.798179,balanced,COMPLETE


#### Tryout best params

In [132]:
log_reg_recall_optimized_model = LogisticRegression(C=0.769946, class_weight='balanced', random_state=random_state)
log_reg_recall_optimized_model.fit(X_train, y_train)
y_pred = log_reg_recall_optimized_model.predict(X_test)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (Recall Optimized)')

              precision    recall  f1-score   support

           0       1.00      0.01      0.02      1036
           1       0.50      1.00      0.67      1035

    accuracy                           0.50      2071
   macro avg       0.75      0.50      0.34      2071
weighted avg       0.75      0.50      0.34      2071



In [133]:
log_reg_f1_optimized_model = LogisticRegression(C=0.315005, class_weight='balanced', random_state=random_state)
log_reg_f1_optimized_model.fit(X_train, y_train)
y_pred = log_reg_f1_optimized_model.predict(X_test)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (F1 Optimized)')

              precision    recall  f1-score   support

           0       1.00      0.01      0.02      1036
           1       0.50      1.00      0.67      1035

    accuracy                           0.50      2071
   macro avg       0.75      0.50      0.34      2071
weighted avg       0.75      0.50      0.34      2071



### Ridge

In [134]:
# We can tune alpha and class weight

#### Recall optimized

In [135]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_recall_optuna_results)

[I 2026-05-27 19:19:04,830] A new study created in memory with name: no-name-4e527e6a-e1fc-44a8-a2af-22e2ec656a87
[I 2026-05-27 19:19:04,967] Trial 0 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.9160121799545452, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:19:05,087] Trial 1 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.5774180563193767, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:19:05,209] Trial 2 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.05390141718460329, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:19:05,316] Trial 3 finished with value: 0.8597323473365851 and parameters: {'alpha': 0.25242539586269463, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8597323473365851.
[I 2026-05-27 19:19:05,426] Trial 4 finished with value: 0.8597323473365851 and paramete

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
0,0,0.859732,2026-05-27 19:19:04.831748,2026-05-27 19:19:04.967681,0 days 00:00:00.135933,0.916012,balanced,COMPLETE
1,1,0.859732,2026-05-27 19:19:04.967681,2026-05-27 19:19:05.087496,0 days 00:00:00.119815,0.577418,balanced,COMPLETE
2,2,0.859732,2026-05-27 19:19:05.087496,2026-05-27 19:19:05.209322,0 days 00:00:00.121826,0.053901,balanced,COMPLETE
3,3,0.859732,2026-05-27 19:19:05.209322,2026-05-27 19:19:05.316473,0 days 00:00:00.107151,0.252425,balanced,COMPLETE
4,4,0.859732,2026-05-27 19:19:05.316473,2026-05-27 19:19:05.426937,0 days 00:00:00.110464,0.215863,None,COMPLETE


#### Tryout best params

In [136]:
ridge_recall_optimized_model = RidgeClassifier(alpha=0.039558, class_weight='balanced', random_state=random_state)
ridge_recall_optimized_model.fit(X_train, y_train)
y_pred = ridge_recall_optimized_model.predict(X_test)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.53123e-24): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


#### F1 optimized

In [137]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_f1_optuna_results)

[I 2026-05-27 19:19:17,382] A new study created in memory with name: no-name-a33ed796-ff6d-4668-b1ab-eceffc6e658f
[I 2026-05-27 19:19:17,507] Trial 0 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.9025321723490667, 'class_weight': None}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:19:17,628] Trial 1 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.7712048527664856, 'class_weight': None}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:19:17,749] Trial 2 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.3289018708241408, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:19:17,901] Trial 3 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.4290513421772031, 'class_weight': None}. Best is trial 0 with value: 0.8769665578707571.
[I 2026-05-27 19:19:18,019] Trial 4 finished with value: 0.8769665578707571 and parameters: {'alpha': 0.5176

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
0,0,0.876967,2026-05-27 19:19:17.382329,2026-05-27 19:19:17.507709,0 days 00:00:00.125380,0.902532,None,COMPLETE
1,1,0.876967,2026-05-27 19:19:17.507709,2026-05-27 19:19:17.628886,0 days 00:00:00.121177,0.771205,None,COMPLETE
2,2,0.876967,2026-05-27 19:19:17.628886,2026-05-27 19:19:17.749825,0 days 00:00:00.120939,0.328902,balanced,COMPLETE
3,3,0.876967,2026-05-27 19:19:17.749825,2026-05-27 19:19:17.901084,0 days 00:00:00.151259,0.429051,None,COMPLETE
4,4,0.876967,2026-05-27 19:19:17.901084,2026-05-27 19:19:18.019480,0 days 00:00:00.118396,0.517645,balanced,COMPLETE


#### Tryout best params

In [138]:
ridge_f1_optimized_model = RidgeClassifier(alpha=0.364826, class_weight='balanced', random_state=random_state)
ridge_f1_optimized_model.fit(X_train, y_train)
y_pred = ridge_f1_optimized_model.predict(X_test)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.20156e-23): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


### Lasso Logistic Regression

In [ ]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [ ]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15, n_jobs=-1)

lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_recall_optuna_results)

[I 2026-05-27 19:29:30,572] A new study created in memory with name: no-name-7bb9c622-251d-46b7-b9ce-0c9386097b97


In [ ]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_f1_optuna_results)

[I 2026-05-27 19:24:07,766] A new study created in memory with name: no-name-557a3f05-bb26-47bc-9871-68116dfe3509
[I 2026-05-27 19:24:08,373] Trial 0 finished with value: 0.04726049275011109 and parameters: {'C': 0.4960233912574134, 'Class_weight': None}. Best is trial 0 with value: 0.04726049275011109.
[I 2026-05-27 19:24:08,969] Trial 1 finished with value: 0.04301554501010194 and parameters: {'C': 0.5885935362159502, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.04726049275011109.
[I 2026-05-27 19:24:09,562] Trial 2 finished with value: 0.04301554501010194 and parameters: {'C': 0.7014105884765477, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.04726049275011109.
[I 2026-05-27 19:24:10,148] Trial 3 finished with value: 0.04726049275011109 and parameters: {'C': 0.1249700890598025, 'Class_weight': None}. Best is trial 0 with value: 0.04726049275011109.
[I 2026-05-27 19:24:10,740] Trial 4 finished with value: 0.04726049275011109 and parameters: {'C': 0.419929372

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
0,0,0.04726,2026-05-27 19:24:07.768325,2026-05-27 19:24:08.373268,0 days 00:00:00.604943,0.496023,None,COMPLETE
3,3,0.04726,2026-05-27 19:24:09.562922,2026-05-27 19:24:10.148980,0 days 00:00:00.586058,0.124970,None,COMPLETE
5,5,0.04726,2026-05-27 19:24:10.740207,2026-05-27 19:24:11.286398,0 days 00:00:00.546191,0.027060,None,COMPLETE
4,4,0.04726,2026-05-27 19:24:10.148980,2026-05-27 19:24:10.740207,0 days 00:00:00.591227,0.419929,None,COMPLETE
28,28,0.04726,2026-05-27 19:24:24.614084,2026-05-27 19:24:25.218892,0 days 00:00:00.604808,0.651292,None,COMPLETE


#### Tryout best params

In [148]:
lasso_recall_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.248927, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_recall_optimized_model.fit(X_train, y_train)
y_pred = lasso_recall_optimized_model.predict(X_test)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.50      0.94      0.65      1036
           1       0.47      0.06      0.10      1035

    accuracy                           0.50      2071
   macro avg       0.48      0.50      0.38      2071
weighted avg       0.48      0.50      0.38      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [146]:
lasso_f1_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.496023, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_f1_optimized_model.fit(X_train, y_train)
y_pred = lasso_f1_optimized_model.predict(X_test)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.50      0.94      0.65      1036
           1       0.47      0.06      0.10      1035

    accuracy                           0.50      2071
   macro avg       0.48      0.50      0.38      2071
weighted avg       0.48      0.50      0.38      2071



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Conclusions

In [149]:
pd.concat(classification_reports, keys=classification_report_keys)

precision    recall  \
Baseline Logistic Regression            0              1.000000  0.007722   
                                        1              0.501697  1.000000   
                                        accuracy       0.503621  0.503621   
                                        macro avg      0.750848  0.503861   
                                        weighted avg   0.750969  0.503621   
Baseline Ridge Classifier               0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Baseline L1 (Lasso) Logistic Regression 0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Logistic Regression (Recall Optimized)  0              1.000000  0.007722   
                                        1              0.501697  1.000000   
                                        accuracy       0.503621  0.503621   
                                        macro avg      0.750848  0.503861   
                                        weighted avg   0.750969  0.503621   
Logistic Regression (F1 Optimized)      0              1.000000  0.007722   
                                        1              0.501697  1.000000   
                                        accuracy       0.503621  0.503621   
                                        macro avg      0.750848  0.503861   
                                        weighted avg   0.750969  0.503621   
Ridge Classifier (Recall Optimized)     0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Ridge Classifier (F1 Optimized)         0              0.880839  0.891892   
                                        1              0.890411  0.879227   
                                        accuracy       0.885563  0.885563   
                                        macro avg      0.885625  0.885559   
                                        weighted avg   0.885623  0.885563   
Lasso Classifier (F1 Optimized)         0              0.498202  0.936293   
                                        1              0.467742  0.056039   
                                        accuracy       0.496379  0.496379   
                                        macro avg      0.482972  0.496166   
                                        weighted avg   0.482980  0.496379   
Lasso Classifier (Recall Optimized)     0              0.498202  0.936293   
                                        1              0.467742  0.056039   
                                        accuracy       0.496379  0.496379   
                                        macro avg      0.482972  0.496166   
                                        weighted avg   0.482980  0.496379   

                                                      f1-score      support  
Baseline Logistic Regression            0             0.015326  1036.000000  
                                        1             0.668173  1035.000000  
                                        accuracy      0.503621     0.503621  
                                        macro avg     0.341749  2071.000000  
                                        weighted avg  0.341592  2071.000000  
Baseline Ridge Classifier               0  

The lasso and ridge both overcome poor initial performance of the logistic regression model

The logistic reg performance could have been poor due to severe multicollinearity or imbalanced feature scaling